# Proyecto VPC: Ataque Adversarial Real Contra YOLOv8-OBB

Este notebook reemplaza la prueba de texturas fijas. La textura se optimiza por gradiente contra el forward diferenciable de YOLOv8-OBB.

Reglas del experimento:

- La textura es un tensor optimizable `theta` con `requires_grad=True`.
- YOLO permanece congelado; no se entrenan sus pesos.
- La composicion de textura usa torch/kornia; no PIL en el camino de gradiente.
- La textura se aplica solo sobre la huella del objeto.
- Primero se valida un objeto; despues se prueba textura universal.

## 1. Entorno, Drive E Imports

In [ ]:
!pip install -q ultralytics kornia

from google.colab import drive
drive.mount('/content/drive')

import json
import random
from pathlib import Path

import kornia.augmentation as K
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import ultralytics
from PIL import Image
from ultralytics import YOLO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

PROJECT_ROOT = Path('/content/drive/MyDrive/Proyecto-VPC')
PREDICTIONS_JSON = PROJECT_ROOT / 'results/digital/baseline_yolo_obb/predictions.json'
DOTA_ROOT = PROJECT_ROOT / 'data/raw/dota_v1'
OUTPUT_DIR = PROJECT_ROOT / 'results/digital/adversarial_gradient'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 1024
WORK_IMAGE_PATH = Path('/content/attack_input_1024.jpg')
TARGET_NAMES = {'small vehicle', 'large vehicle'}

print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| device', DEVICE)
print('ultralytics', ultralytics.__version__)
print('predictions:', PREDICTIONS_JSON, PREDICTIONS_JSON.exists())

## 2. Cargar YOLOv8-OBB Congelado

In [ ]:
yolo = YOLO('yolov8n-obb.pt')
model = yolo.model.to(DEVICE).eval()

for p in model.parameters():
    p.requires_grad_(False)

CLASS_NAMES = yolo.names
TARGET_IDS = [i for i, name in CLASS_NAMES.items() if name in TARGET_NAMES]

print('Clases:', CLASS_NAMES)
print('Target names:', TARGET_NAMES)
print('Target ids:', TARGET_IDS)
assert TARGET_IDS, 'No encontre IDs para las clases objetivo.'

## 3. Forward Diferenciable Y Split De Predicciones

Validamos que el tensor crudo tenga layout `[B, 4+nc+1, A]`: 4 caja, `nc` clases y 1 angulo.

In [ ]:
def _find_prediction_tensor(out, nc):
    expected = 4 + nc + 1
    if torch.is_tensor(out):
        if out.ndim == 3 and (out.shape[1] == expected or out.shape[2] == expected):
            return out
        raise TypeError(f'Tensor inesperado: shape={tuple(out.shape)}, expected={expected}')

    if isinstance(out, (tuple, list)):
        for item in out:
            try:
                return _find_prediction_tensor(item, nc)
            except TypeError:
                continue

    raise TypeError(f'No pude encontrar tensor de predicciones en {type(out)}')


def yolo_forward_raw(image_bchw):
    out = model(image_bchw)
    raw = _find_prediction_tensor(out, len(CLASS_NAMES))
    expected = 4 + len(CLASS_NAMES) + 1
    if raw.shape[1] != expected and raw.shape[2] == expected:
        raw = raw.transpose(1, 2).contiguous()
    return raw


def split_predictions(raw):
    nc = len(CLASS_NAMES)
    expected = 4 + nc + 1
    if raw.ndim != 3:
        raise ValueError(f'raw debe ser 3D, recibido {tuple(raw.shape)}')
    if raw.shape[1] != expected and raw.shape[2] == expected:
        raw = raw.transpose(1, 2).contiguous()
    if raw.shape[1] != expected:
        raise ValueError(f'Layout no soportado: {tuple(raw.shape)}, expected C={expected}')

    boxes = raw[:, 0:4, :]
    cls_raw = raw[:, 4:4 + nc, :]
    angle = raw[:, 4 + nc:4 + nc + 1, :]

    # Ultralytics suele devolver confianzas post-sigmoide; si son logits, corregimos.
    cls = cls_raw.sigmoid() if (cls_raw.detach().min() < 0 or cls_raw.detach().max() > 1) else cls_raw
    return boxes, cls, angle


dummy = torch.rand(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    raw = yolo_forward_raw(dummy)
boxes, cls, angle = split_predictions(raw)

print('raw.shape:', tuple(raw.shape))
print('boxes:', tuple(boxes.shape), 'cls:', tuple(cls.shape), 'angle:', tuple(angle.shape))
print('cls range:', float(cls.min()), float(cls.max()))
assert raw.shape[1] == 4 + len(CLASS_NAMES) + 1, 'raw no esta en layout [B,4+nc+1,A]'
assert cls.min() >= 0 and cls.max() <= 1, 'cls debe estar post-sigmoide'

## 4. Elegir Imagen DOTA Y Objeto Objetivo

Usa `predictions.json` si existe. Si no, busca imagenes dentro de `data/raw/dota_v1`. YOLO elige el vehiculo objetivo en una copia 1024x1024 para que la escala del bbox coincida con el forward diferenciable.

In [ ]:
def choose_dota_image():
    if PREDICTIONS_JSON.exists():
        payload = json.loads(PREDICTIONS_JSON.read_text(encoding='utf-8'))
        for record in payload.get('records', []):
            has_target = any(det.get('class_name') in TARGET_NAMES for det in record.get('detections', []))
            path = Path(record.get('image_path', ''))
            if has_target and path.exists():
                return path

    candidates = []
    for ext in ('*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff'):
        candidates.extend(DOTA_ROOT.rglob(ext))
    assert candidates, f'No encontre imagenes en {DOTA_ROOT}'
    return sorted(candidates)[0]


def prepare_work_image(path, size=IMG_SIZE):
    img = Image.open(path).convert('RGB').resize((size, size))
    img.save(WORK_IMAGE_PATH, quality=95)
    arr = np.array(img)
    tensor = torch.from_numpy(arr).float().permute(2, 0, 1) / 255.0
    return tensor.unsqueeze(0)


IMAGE_PATH = choose_dota_image()
image = prepare_work_image(IMAGE_PATH).to(DEVICE)

det = yolo.predict(str(WORK_IMAGE_PATH), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
assert det.obb is not None and len(det.obb) > 0, 'YOLO no detecto nada; usa otra imagen'

xyxy = det.obb.xyxy.detach().cpu().numpy()
cls_ids = det.obb.cls.detach().cpu().numpy().astype(int)
areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
order = np.argsort(-areas)
pick = next((i for i in order if cls_ids[i] in TARGET_IDS), None)
assert pick is not None, 'YOLO no encontro vehiculos objetivo en esta imagen'

X1, Y1, X2, Y2 = xyxy[pick]
box = (float(X1), float(Y1), float(X2), float(Y2))

print('Imagen base:', IMAGE_PATH)
print('Imagen de trabajo:', WORK_IMAGE_PATH)
print(f'Objeto objetivo: clase={CLASS_NAMES[cls_ids[pick]]} bbox=({X1:.1f},{Y1:.1f},{X2:.1f},{Y2:.1f}) area={areas[pick]:.1f}')

plt.figure(figsize=(7, 7))
plt.imshow(image[0].permute(1, 2, 0).detach().cpu())
plt.gca().add_patch(plt.Rectangle((X1, Y1), X2-X1, Y2-Y1, fill=False, ec='red', lw=2))
plt.title('Objeto objetivo')
plt.axis('off')
plt.show()

## 5. Textura Optimizable Y Composicion Diferenciable

In [ ]:
PATCH_H, PATCH_W = 128, 128
theta = None
optimizer = None


def reset_texture(init_scale=0.5, lr=0.1):
    global theta, optimizer
    theta = torch.randn(3, PATCH_H, PATCH_W, device=DEVICE) * init_scale
    theta.requires_grad_(True)
    optimizer = torch.optim.Adam([theta], lr=lr)
    return theta


def get_texture():
    return torch.sigmoid(theta)


def clamp_box(box, width=IMG_SIZE, height=IMG_SIZE):
    x1, y1, x2, y2 = [int(round(v)) for v in box]
    x1 = max(0, min(width - 1, x1))
    y1 = max(0, min(height - 1, y1))
    x2 = max(x1 + 1, min(width, x2))
    y2 = max(y1 + 1, min(height, y2))
    return x1, y1, x2, y2


eot_geom = K.RandomAffine(degrees=15.0, scale=(0.85, 1.15), p=1.0)
eot_color = K.ColorJitter(brightness=0.25, contrast=0.25, p=1.0)


def apply_eot(patch):
    out = eot_geom(patch)
    out = eot_color(out)
    out = out + torch.randn_like(out) * 0.01
    return out.clamp(0.0, 1.0)


def place_texture(image_bchw, texture_chw, box, eot=True):
    b, c, h, w = image_bchw.shape
    x1, y1, x2, y2 = clamp_box(box, w, h)
    bw, bh = x2 - x1, y2 - y1

    patch = texture_chw.unsqueeze(0)
    if eot:
        patch = apply_eot(patch)
    patch = F.interpolate(patch, size=(bh, bw), mode='bilinear', align_corners=False)

    canvas = torch.zeros_like(image_bchw)
    mask = torch.zeros(1, 1, h, w, device=image_bchw.device, dtype=image_bchw.dtype)
    canvas[:, :, y1:y2, x1:x2] = patch
    mask[:, :, y1:y2, x1:x2] = 1.0
    return image_bchw * (1.0 - mask) + canvas * mask


reset_texture()
print('theta:', tuple(theta.shape), '| parametros:', theta.numel())

## 6. Loss Correcto: Max Confianza Dentro De Zona Del Objeto

Usamos `max_inside` porque el promedio diluye la senal. El loss inicial deberia acercarse a la confianza de YOLO `predict` dentro del objeto.

In [ ]:
def detection_loss(raw, box, return_debug=False, pad_scale=0.5):
    boxes, cls, _ = split_predictions(raw)
    cx, cy = boxes[:, 0, :], boxes[:, 1, :]
    x1, y1, x2, y2 = box

    pad_x = (x2 - x1) * pad_scale
    pad_y = (y2 - y1) * pad_scale
    rx1, ry1 = x1 - pad_x, y1 - pad_y
    rx2, ry2 = x2 + pad_x, y2 + pad_y

    inside = ((cx >= rx1) & (cx <= rx2) & (cy >= ry1) & (cy <= ry2)).float()
    tgt_conf = cls[:, TARGET_IDS, :].amax(dim=1)
    inside_sum = inside.sum()
    masked = tgt_conf.masked_fill(inside <= 0, -1e4)
    loss = masked.max()

    if return_debug:
        valid = tgt_conf[inside > 0]
        debug = {
            'inside_sum': float(inside_sum.detach().cpu()),
            'target_conf_max_global': float(tgt_conf.detach().max().cpu()),
            'target_conf_max_inside': float(loss.detach().cpu()),
            'target_conf_mean_inside': float(valid.mean().detach().cpu()) if valid.numel() else 0.0,
            'cx_range': (float(cx.detach().min().cpu()), float(cx.detach().max().cpu())),
            'cy_range': (float(cy.detach().min().cpu()), float(cy.detach().max().cpu())),
        }
        return loss, debug
    return loss


def tv_loss(texture):
    dh = (texture[:, 1:, :] - texture[:, :-1, :]).abs().mean()
    dw = (texture[:, :, 1:] - texture[:, :, :-1]).abs().mean()
    return dh + dw


PRINTABLE = torch.tensor([
    [0.13, 0.13, 0.13], [0.30, 0.32, 0.27], [0.42, 0.45, 0.38],
    [0.52, 0.52, 0.48], [0.78, 0.78, 0.72], [0.86, 0.86, 0.86],
], device=DEVICE)


def nps_loss(texture):
    pixels = texture.permute(1, 2, 0).reshape(-1, 3)
    distances = (pixels.unsqueeze(1) - PRINTABLE.unsqueeze(0)).pow(2).sum(-1)
    return distances.min(dim=1).values.mean()


with torch.no_grad():
    raw_clean = yolo_forward_raw(image)
    clean_loss, clean_dbg = detection_loss(raw_clean, box, return_debug=True)
print('debug clean:', clean_dbg)
assert clean_dbg['inside_sum'] > 0, 'inside.sum() es 0: revisa box/escala.'

## 7. Sanity Check De Gradiente En Theta

In [ ]:
reset_texture(init_scale=0.5, lr=0.1)
optimizer.zero_grad(set_to_none=True)
texture = get_texture()
adv_test = place_texture(image, texture, box, eot=False)
raw_test = yolo_forward_raw(adv_test)
loss_test, dbg_test = detection_loss(raw_test, box, return_debug=True)
loss_test.backward()
grad_sum = float(theta.grad.abs().sum().detach().cpu()) if theta.grad is not None else 0.0

print('raw.shape:', tuple(raw_test.shape))
print('debug:', dbg_test)
print('L_det:', float(loss_test.detach().cpu()))
print('theta.grad.abs().sum():', grad_sum)

assert dbg_test['inside_sum'] > 0, 'inside.sum() es 0.'
assert grad_sum > 0, 'Gradiente cero en theta.'
optimizer.zero_grad(set_to_none=True)
print('Sanity check OK')

## 8. Ataque De Un Objeto Sin EoT/TV

Primero se prueba la version agresiva. Si esta no funciona, no tiene sentido correr loops largos o universales.

In [ ]:
ITERS_ONE = 500
LR_ONE = 0.10
USE_EOT_ONE = False
LAMBDA_TV_ONE = 0.0

reset_texture(init_scale=0.5, lr=LR_ONE)
history = []

for it in range(1, ITERS_ONE + 1):
    optimizer.zero_grad(set_to_none=True)
    texture = get_texture()
    adv_image = place_texture(image, texture, box, eot=USE_EOT_ONE)
    raw_adv = yolo_forward_raw(adv_image)
    l_det, dbg = detection_loss(raw_adv, box, return_debug=True)
    l_tv = tv_loss(texture)
    loss = l_det + LAMBDA_TV_ONE * l_tv
    loss.backward()
    grad_sum = float(theta.grad.abs().sum().detach().cpu()) if theta.grad is not None else 0.0
    optimizer.step()

    history.append((it, l_det.item(), l_tv.item(), loss.item(), dbg['inside_sum'], grad_sum, dbg['target_conf_max_inside']))

    if it == 1 or it % 25 == 0:
        print(f"it {it:4d} | L_det {l_det.item():.4f} | max_inside {dbg['target_conf_max_inside']:.4f} | inside {dbg['inside_sum']:.0f} | grad {grad_sum:.2e}")

    if dbg['inside_sum'] <= 0:
        raise RuntimeError('inside.sum() es 0 durante entrenamiento')
    if grad_sum <= 0:
        raise RuntimeError('theta.grad es cero durante entrenamiento')

print('L_det inicio:', history[0][1])
print('L_det final:', history[-1][1])

## 9. Curvas Del Ataque De Un Objeto

In [ ]:
iters = np.array([row[0] for row in history])
l_det = np.array([row[1] for row in history])
inside = np.array([row[4] for row in history])
grad = np.array([row[5] for row in history])

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(iters, l_det)
plt.title('L_det max_inside')
plt.grid(alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(iters, inside)
plt.title('inside.sum')
plt.grid(alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(iters, grad)
plt.title('theta.grad.abs().sum')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Delta L_det:', l_det[0] - l_det[-1])

## 10. Evaluacion Con YOLO Real Y NMS

In [ ]:
def count_targets_inside(det, box):
    if det.obb is None:
        return 0, []
    x1, y1, x2, y2 = box
    xyxy = det.obb.xyxy.detach().cpu().numpy()
    cls_ids = det.obb.cls.detach().cpu().numpy().astype(int)
    conf = det.obb.conf.detach().cpu().numpy()
    hits = []
    for bbox, class_id, score in zip(xyxy, cls_ids, conf):
        if class_id not in TARGET_IDS:
            continue
        cx = (bbox[0] + bbox[2]) / 2
        cy = (bbox[1] + bbox[3]) / 2
        if x1 <= cx <= x2 and y1 <= cy <= y2:
            hits.append({'class': CLASS_NAMES[class_id], 'conf': float(score), 'bbox': bbox.tolist()})
    return len(hits), hits


ADV_IMAGE_PATH = OUTPUT_DIR / 'adv_result_one_object.jpg'
TEXTURE_PATH = OUTPUT_DIR / 'texture_one_object.png'

with torch.no_grad():
    final_texture = get_texture()
    adv_image = place_texture(image, final_texture, box, eot=False)

adv_np = (adv_image[0].permute(1, 2, 0).detach().cpu().numpy() * 255).astype(np.uint8)
tex_np = (final_texture.permute(1, 2, 0).detach().cpu().numpy() * 255).astype(np.uint8)
Image.fromarray(adv_np).save(ADV_IMAGE_PATH, quality=95)
Image.fromarray(tex_np).save(TEXTURE_PATH)

det_clean = yolo.predict(str(WORK_IMAGE_PATH), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
det_adv = yolo.predict(str(ADV_IMAGE_PATH), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]

clean_inside_count, clean_hits = count_targets_inside(det_clean, box)
adv_inside_count, adv_hits = count_targets_inside(det_adv, box)

print('Dentro del bbox atacado:')
print('Original:', clean_inside_count, clean_hits[:5])
print('Atacado:', adv_inside_count, adv_hits[:5])
print('Imagen atacada:', ADV_IMAGE_PATH)
print('Textura:', TEXTURE_PATH)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(det_clean.plot()[..., ::-1]); ax[0].set_title('Original YOLO'); ax[0].axis('off')
ax[1].imshow(final_texture.detach().permute(1, 2, 0).cpu()); ax[1].set_title('Textura optimizada'); ax[1].axis('off')
ax[2].imshow(det_adv.plot()[..., ::-1]); ax[2].set_title('Atacado YOLO'); ax[2].axis('off')
plt.show()

## 11. Construir Objetos Para Textura Universal

In [ ]:
MAX_UNIVERSAL_OBJECTS = 80
MIN_BOX_SIDE = 6

def load_attack_tensor_cpu(path, size=IMG_SIZE):
    img = Image.open(path).convert('RGB').resize((size, size))
    arr = np.array(img)
    return torch.from_numpy(arr).float().permute(2, 0, 1).unsqueeze(0) / 255.0


def polygon_to_bbox_scaled(polygon, original_size, size=IMG_SIZE):
    points = np.array(polygon, dtype=np.float32).reshape(-1, 2)
    x1, y1 = points.min(axis=0)
    x2, y2 = points.max(axis=0)
    ow, oh = original_size
    return (float(x1 * size / ow), float(y1 * size / oh), float(x2 * size / ow), float(y2 * size / oh))


assert PREDICTIONS_JSON.exists(), f'Falta predictions.json: {PREDICTIONS_JSON}'
payload = json.loads(PREDICTIONS_JSON.read_text(encoding='utf-8'))
image_cache = {}
objetos = []

for record in payload.get('records', []):
    image_path = Path(record['image_path'])
    if not image_path.exists():
        continue
    with Image.open(image_path) as img_meta:
        original_size = img_meta.size
    if image_path not in image_cache:
        image_cache[image_path] = load_attack_tensor_cpu(image_path)

    for det in record.get('detections', []):
        if det.get('class_name') not in TARGET_NAMES or det.get('polygon') is None:
            continue
        box_i = polygon_to_bbox_scaled(det['polygon'], original_size)
        x1, y1, x2, y2 = box_i
        if (x2 - x1) < MIN_BOX_SIDE or (y2 - y1) < MIN_BOX_SIDE:
            continue
        objetos.append((image_cache[image_path], box_i, str(image_path), det.get('class_name')))
        if len(objetos) >= MAX_UNIVERSAL_OBJECTS:
            break
    if len(objetos) >= MAX_UNIVERSAL_OBJECTS:
        break

print('Objetos universales:', len(objetos))
assert objetos, 'No se construyeron objetos desde predictions.json'
print('Ejemplo:', Path(objetos[0][2]).name, objetos[0][3], objetos[0][1])

## 12. Loop Universal

Usa una sola textura sobre multiples objetos. Primero puedes dejar `USE_EOT_UNIV=False`; cuando funcione, cambia a `True`.

In [ ]:
UNIVERSAL_ITERS = 800
UNIVERSAL_LR = 0.03
USE_EOT_UNIV = False
LAMBDA_TV_UNIV = 0.0

theta_univ = theta.detach().clone().requires_grad_(True)
optimizer_univ = torch.optim.Adam([theta_univ], lr=UNIVERSAL_LR)

def get_texture_universal():
    return torch.sigmoid(theta_univ)

history_univ = []

for it in range(1, UNIVERSAL_ITERS + 1):
    img_cpu, box_i, image_name, cls_name = random.choice(objetos)
    img_t = img_cpu.to(DEVICE)
    optimizer_univ.zero_grad(set_to_none=True)

    texture = get_texture_universal()
    adv = place_texture(img_t, texture, box_i, eot=USE_EOT_UNIV)
    raw_adv = yolo_forward_raw(adv)
    l_det, dbg = detection_loss(raw_adv, box_i, return_debug=True)
    l_tv = tv_loss(texture)
    loss = l_det + LAMBDA_TV_UNIV * l_tv
    loss.backward()
    grad_sum = float(theta_univ.grad.abs().sum().detach().cpu()) if theta_univ.grad is not None else 0.0
    optimizer_univ.step()

    history_univ.append((it, l_det.item(), l_tv.item(), loss.item(), dbg['inside_sum'], grad_sum, image_name, cls_name))
    if it == 1 or it % 100 == 0:
        print(f"it {it:4d} | L_det {l_det.item():.4f} | inside {dbg['inside_sum']:.0f} | grad {grad_sum:.2e} | {Path(image_name).name} {cls_name}")

    if dbg['inside_sum'] <= 0 or grad_sum <= 0:
        raise RuntimeError('Loop universal sin inside o sin gradiente')

theta = theta_univ
optimizer = optimizer_univ
print('Loop universal terminado')
print('L_det inicio:', history_univ[0][1], '| L_det final:', history_univ[-1][1])

## 13. Exportar Textura

In [ ]:
export_path = OUTPUT_DIR / 'textura_adversarial_universal.png'
texture_np = (get_texture().detach().permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
Image.fromarray(texture_np).resize((1024, 1024), Image.NEAREST).save(export_path)
print('Guardada:', export_path)